## 步骤1: 打开屏幕配置


In [ ]:
%%bash
systemsettings kcm_kscreen


## 步骤3: 安装 WhiteSur 主题


In [ ]:
%%bash
is_theme_installed() {
	local theme_name="$1"
	case "$theme_name" in
	WhiteSur-icon-theme)
		[[ -d "$HOME/.local/share/icons/WhiteSur" ]] || [[ -d "$HOME/.icons/WhiteSur" ]]
		;;
	WhiteSur-kde)
		[[ -d "$HOME/.local/share/plasma/desktoptheme/WhiteSur" ]]
		;;
	WhiteSur-cursors)
		[[ -d "$HOME/.local/share/icons/WhiteSur-cursors" ]] || [[ -d "$HOME/.icons/WhiteSur-cursors" ]]
		;;
	*)
		return 1
		;;
	esac
}

install_theme() {
	local git_url="$1"
	local themes_dir="${2:-$HOME/Downloads/.themes}"
	local theme_name
	theme_name=$(basename "$git_url" .git)

	if is_theme_installed "$theme_name"; then
		echo "  ✓ $theme_name already installed, skipping"
		return 0
	fi

	mkdir -p "$themes_dir"
	cd "$themes_dir" || return 1

	local theme_path="$themes_dir/$theme_name"

	if [[ ! -d "$theme_path" ]]; then
		git clone "$git_url" &>/dev/null && echo "  ✓ Cloned $theme_name"
	fi

	cd "$theme_path" || return 1

	if [[ "$theme_name" == "WhiteSur-cursors" && -f "build.sh" ]]; then
		bash build.sh &>/dev/null && echo "  ✓ Built cursors"
	fi

	if [[ -f "install.sh" ]]; then
		bash install.sh && echo "  ✓ Installed $theme_name"
	fi

	rm -rf "$theme_path" && echo "  ✓ Cleaned $theme_name"
}
urls="https://github.com/vinceliuice/WhiteSur-icon-theme.git
https://github.com/vinceliuice/WhiteSur-kde.git
https://github.com/vinceliuice/WhiteSur-cursors.git"

echo "🎨 Installing themes..."
while IFS= read -r url; do
	url=$(echo "$url" | xargs)
	[[ -z "$url" ]] && continue
	echo "Installing theme from: $url"
	install_theme "$url"
done <<<"$urls"
rm -rf "$HOME/Downloads/.themes"
echo "✓ All themes processed"


## 步骤4: 打开外观设置


In [ ]:
%%bash
kcmshell6 kcm_lookandfeel


## 步骤5: 禁用蓝牙


In [ ]:
%%bash
systemctl is-active --quiet bluetooth && sudo systemctl disable --now bluetooth || true


## 步骤6: 配置壁纸和面板


In [ ]:
%%bash
配置时钟显示秒,并使用12小时制
#!/bin/bash
config_file="$HOME/.config/plasma-org.kde.plasma.desktop-appletsrc"

# 查找 digitalclock 的 containment 和 applet ID
while IFS= read -r line; do
    if [[ "$line" =~ ^\[Containments\]\[([0-9]+)\]\[Applets\]\[([0-9]+)\]$ ]]; then
        containment_id="${BASH_REMATCH[1]}"
        applet_id="${BASH_REMATCH[2]}"
        
        # 检查是否为 digitalclock 插件
        plugin=$(kreadconfig6 --file "$config_file" \
            --group "Containments" --group "$containment_id" \
            --group "Applets" --group "$applet_id" \
            --key "plugin")
        
        if [[ "$plugin" == "org.kde.plasma.digitalclock" ]]; then
            echo "找到 digitalclock: Containment=$containment_id, Applet=$applet_id"
            
            # 配置时钟显示秒数和12小时制
            kwriteconfig6 --file "$config_file" \
                --group "Containments" --group "$containment_id" \
                --group "Applets" --group "$applet_id" \
                --group "Configuration" --group "Appearance" \
                --key "showSeconds" "Always"
            
            kwriteconfig6 --file "$config_file" \
                --group "Containments" --group "$containment_id" \
                --group "Applets" --group "$applet_id" \
                --group "Configuration" --group "Appearance" \
                --key "use24hFormat" "2"
            
            echo "✓ 已配置 digitalclock"
        fi
    fi
done < "$config_file"



In [ ]:
%%bash
# 同理,查找[Containments][xx] xx为动态生成,的location=0,plugin=org.kde.plasma.folder,并设置wallpaperplugin=org.kde.potd
# 然后配置[Containments][26][Wallpaper][org.kde.potd][General]的Provider=bing
#!/bin/bash
config_file="$HOME/.config/plasma-org.kde.plasma.desktop-appletsrc"

# 查找桌面容器 (location=0 且 plugin=org.kde.plasma.folder)
grep -oP '^\[Containments\]\[\K\d+(?=\])' "$config_file" | sort -u | while read -r c_id; do
    location=$(kreadconfig6 --file "$config_file" --group "Containments" --group "$c_id" --key "location")
    plugin=$(kreadconfig6 --file "$config_file" --group "Containments" --group "$c_id" --key "plugin")
    
    if [[ "$location" == "0" && "$plugin" == "org.kde.plasma.folder" ]]; then
        echo "找到桌面容器: Containment=$c_id"
        
        # 设置壁纸插件为 org.kde.potd
        kwriteconfig6 --file "$config_file" \
            --group "Containments" --group "$c_id" \
            --key "wallpaperplugin" "org.kde.potd"
        
        # 配置 Bing 每日壁纸
        kwriteconfig6 --file "$config_file" \
            --group "Containments" --group "$c_id" \
            --group "Wallpaper" --group "org.kde.potd" --group "General" \
            --key "Provider" "bing"
        
        echo "✓ 已配置 Bing 每日壁纸"
    fi
done



In [ ]:
%%bash
# 参考digitalclock
# plugin=org.kde.plasma.icontasks
# 设置launchers
# [Containments][29][Applets][30][Configuration][General]
# launchers=preferred://filemanager,applications:systemsettings.desktop,applications:org.keepassxc.KeePassXC.desktop,applications:org.cryptomator.Cryptomator.desktop,applications:org.kde.konsole.desktop,applications:code.desktop,applications:google-chrome.desktop,applications:firefox.desktop,applications:Clash Verge.desktop
#!/bin/bash
config_file="$HOME/.config/plasma-org.kde.plasma.desktop-appletsrc"

# 查找 icontasks 的 containment 和 applet ID
while IFS= read -r line; do
    if [[ "$line" =~ ^\[Containments\]\[([0-9]+)\]\[Applets\]\[([0-9]+)\]$ ]]; then
        containment_id="${BASH_REMATCH[1]}"
        applet_id="${BASH_REMATCH[2]}"
        
        # 检查是否为 icontasks 插件
        plugin=$(kreadconfig6 --file "$config_file" \
            --group "Containments" --group "$containment_id" \
            --group "Applets" --group "$applet_id" \
            --key "plugin")
        
        if [[ "$plugin" == "org.kde.plasma.icontasks" ]]; then
            echo "找到 icontasks: Containment=$containment_id, Applet=$applet_id"
            
            # 配置启动器列表
            launchers="preferred://filemanager,applications:systemsettings.desktop,applications:org.keepassxc.KeePassXC.desktop,applications:org.cryptomator.Cryptomator.desktop,applications:org.kde.konsole.desktop,applications:code.desktop,applications:google-chrome.desktop,applications:firefox.desktop,applications:Clash Verge.desktop"
            
            kwriteconfig6 --file "$config_file" \
                --group "Containments" --group "$containment_id" \
                --group "Applets" --group "$applet_id" \
                --group "Configuration" --group "General" \
                --key "launchers" "$launchers"
            
            echo "✓ 已配置 icontasks 启动器"
        fi
    fi
done < "$config_file"


## 步骤7: 安装字体


In [ ]:
%%bash
fonts="inter-font
adobe-source-han-sans-otc-fonts
adobe-source-han-serif-otc-fonts 
noto-fonts
noto-fonts-cjk
noto-fonts-emoji
ttf-dejavu
ttf-liberation
wqy-microhei
wqy-zenhei
adobe-source-han-sans-cn-fonts
adobe-source-han-serif-cn-fonts
ttf-fira-code
ttf-roboto"

echo "✓ Installing fonts..."
for font in $fonts; do
	sudo pacman -Sy --needed --noconfirm "$font" &>/dev/null && echo "  ✓ $font installed"
done


## 步骤8: 配置系统字体


In [ ]:
%%bash
config_file="$HOME/.config/kdeglobals"
kwriteconfig6 --file "$config_file" --group "General" --key "font" "Source Han Sans CN,10,-1,5,316,0,0,0,0,0,0,0,0,0,0,1,Normal"
kwriteconfig6 --file "$config_file" --group "General" --key "menuFont" "Source Han Sans CN,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1"
kwriteconfig6 --file "$config_file" --group "General" --key "smallestReadableFont" "Source Han Sans CN,8,-1,5,400,0,0,0,0,0,0,0,0,0,0,1"
kwriteconfig6 --file "$config_file" --group "General" --key "toolBarFont" "Source Han Sans CN,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1"
kwriteconfig6 --file "$config_file" --group "General" --key "fixed" "Source Han Sans CN,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1"
kwriteconfig6 --file "$config_file" --group "WM" --key "activeFont" "Source Han Sans CN,10,-1,5,400,0,0,0,0,0,0,0,0,0,0,1"


## 步骤9: 配置鼠标和键盘


In [ ]:
%%bash
config_file="$HOME/.config/kcminputrc"
kwriteconfig6 --file "$config_file" --group "Libinput" --group "4152" --group "5898" --group "SteelSeries SteelSeries Rival 100 Dell China" --key "PointerAcceleration" "1"
kwriteconfig6 --file "$config_file" --group "Keyboard" --key "NumLock" "0"


## 步骤10: 配置 KWin 窗口管理器


In [ ]:
%%bash
echo "✓ Configuring KWin..."
config_file="$HOME/.config/kwinrc"
kwriteconfig6 --file "$config_file" --group "Effect-overview" --key "BorderActivate" "1,7"
kwriteconfig6 --file "$config_file" --group "ElectricBorders" --key "BottomLeft" "KRunner"
kwriteconfig6 --file "$config_file" --group "ElectricBorders" --key "TopLeft" "ApplicationLauncher"
kwriteconfig6 --file "$config_file" --group "TabBox" --key "LayoutName" "big_icons"
kwriteconfig6 --file "$config_file" --group "General" --key "FreeFloating" --type bool true
kwriteconfig6 --file "$config_file" --group "General" --key "historyBehavior" "ImmediateCompletion"


## 步骤11: 配置动画效果


In [ ]:
%%bash
config_file="$HOME/.config/kwinrc"

declare -A animations=(
	["mouseclickEnabled"]="true"
	["trackmouseEnabled"]="true"
	["contrastEnabled"]="true"
	["blurEnabled"]="true"
	["fallapartEnabled"]="false"
	["mousemarkEnabled"]="true"
	["translucencyEnabled"]="true"
	["wobblywindowsEnabled"]="true"
	["magiclampEnabled"]="true"
	["squashEnabled"]="false"
	["diminactiveEnabled"]="true"
	["glideEnabled"]="true"
	["scaleEnabled"]="false"
)

for key in "${!animations[@]}"; do
	kwriteconfig6 --file "$config_file" --group "Plugins" --key "$key" --type bool "${animations[$key]}"
done


## 步骤12: 配置屏幕锁定


In [ ]:
%%bash
echo "✓ Configuring screen locker..."
config_file="$HOME/.config/kscreenlockerrc"
kwriteconfig6 --file "$config_file" --group "Daemon" --key "Timeout" "0"
kwriteconfig6 --file "$config_file" --group "Daemon" --key "LockGrace" "900"
kwriteconfig6 --file "$config_file" --group "Daemon" --key "RequirePassword" "false"
kwriteconfig6 --file "$config_file" --group "Daemon" --key "Autolock" "false"


## 步骤13: 配置活动管理器


In [ ]:
%%bash
echo "✓ Configuring activity manager..."
config_file="$HOME/.config/kactivitymanagerd-pluginsrc"
kwriteconfig6 --file "$config_file" --group "Plugin-org.kde.ActivityManager.Resources.Scoring" --key "keep-history-for" "1"


## 步骤14: 配置 KRunner


In [ ]:
%%bash
config_file="$HOME/.config/krunnerrc"
kwriteconfig6 --file "$config_file" --group "General" --key "FreeFloating" --type bool "true"


## 步骤15: 重启 Plasma 和 KWin


In [ ]:
%%bash
kquitapp6 plasmashell && kstart plasmashell &
qdbus6 org.kde.KWin /KWin reconfigure
echo "✓ KDE configuration completed."
